# Experiment: PI1M Endpoint Pipeline Logs

이 노트북은 CLI 파이프라인을 그대로 실행하고 로그/산출물을 확인하기 위한 런너입니다.
- preprocess -> train -> eval(constrained/unconstrained)
- 생성 로그는 `outputs/notebook_logs/` 아래에 저장됩니다.
- 기본은 `DEBUG=1` (빠른 스모크 실행)입니다.


In [ ]:
from pathlib import Path
import subprocess
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "scripts" / "run_preprocess.sh").exists():
    # notebook을 다른 cwd에서 열었을 때 자동 보정
    ROOT = Path('/home/kiket/Desktop/test/MY_PAPER_RELATED/selfies-psmiles')

LOG_DIR = ROOT / 'outputs' / 'notebook_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
print('ROOT =', ROOT)
print('LOG_DIR =', LOG_DIR)


def run_cmd(cmd: str, log_name: str):
    log_path = LOG_DIR / log_name
    print(f"\n[RUN] {cmd}")
    proc = subprocess.run(
        cmd,
        shell=True,
        cwd=ROOT,
        text=True,
        capture_output=True,
    )
    log_text = "".join([
        f"$ {cmd}\n",
        "\n[STDOUT]\n",
        proc.stdout,
        "\n[STDERR]\n",
        proc.stderr,
    ])
    log_path.write_text(log_text, encoding='utf-8')
    print(f"returncode = {proc.returncode}")
    print(f"log = {log_path}")
    if proc.returncode != 0:
        print(proc.stdout[-2000:])
        print(proc.stderr[-2000:])
        raise RuntimeError(f"command failed: {cmd}")
    return log_path


## 1) Preprocess (Debug)

전체 PI1M 실행이 아니라 빠른 검증을 위해 `DEBUG=1`로 동작합니다.
전체 실행은 `DEBUG=1`을 제거해서 같은 명령을 실행하면 됩니다.


In [ ]:
pre_log = run_cmd(
    "DEBUG=1 bash scripts/run_preprocess.sh",
    "preprocess_debug.log",
)
pre_log


In [ ]:
summary_path = ROOT / 'outputs' / 'pi1m_endpoint_dataset' / 'dataset_summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
summary


## 2) Train Baseline (Debug)


In [ ]:
train_log = run_cmd(
    "DEBUG=1 bash scripts/run_train.sh configs/pi1m_endpoint_baseline.yaml",
    "train_debug.log",
)
train_log


In [ ]:
runs = sorted((ROOT / 'outputs' / 'experiments').glob('pi1m_endpoint_baseline_*'))
assert runs, 'no training run found'
RUN_DIR = runs[-1]
RUN_DIR


In [ ]:
train_summary = json.loads((RUN_DIR / 'train_summary.json').read_text(encoding='utf-8'))
history = json.loads((RUN_DIR / 'train_history.json').read_text(encoding='utf-8'))
print(train_summary)
pd.DataFrame(history).tail(10)


## 3) Evaluate (Constrained)


In [ ]:
ckpt = RUN_DIR / 'checkpoints' / 'best.pt'
assert ckpt.exists(), ckpt

constrained_log = run_cmd(
    f"bash scripts/run_eval.sh configs/pi1m_endpoint_baseline.yaml {ckpt} constrained",
    "eval_constrained.log",
)
constrained_log


In [ ]:
eval_runs = sorted((ROOT / 'outputs' / 'eval').glob('best_*'))
assert eval_runs, 'no eval outputs found'
EVAL_CONSTRAINED_DIR = eval_runs[-1]
constrained_metrics = json.loads((EVAL_CONSTRAINED_DIR / 'metrics.json').read_text(encoding='utf-8'))
EVAL_CONSTRAINED_DIR, constrained_metrics


## 4) Evaluate Ablation (Unconstrained)


In [ ]:
unconstrained_log = run_cmd(
    f"bash scripts/run_eval.sh configs/pi1m_endpoint_ablation.yaml {ckpt} unconstrained",
    "eval_unconstrained.log",
)
unconstrained_log


In [ ]:
eval_runs2 = sorted((ROOT / 'outputs' / 'eval').glob('best_*'))
EVAL_UNCONSTRAINED_DIR = eval_runs2[-1]
unconstrained_metrics = json.loads((EVAL_UNCONSTRAINED_DIR / 'metrics.json').read_text(encoding='utf-8'))

pd.DataFrame([
    {"mode": "constrained", **constrained_metrics},
    {"mode": "unconstrained", **unconstrained_metrics},
])[[
    'mode',
    'endpoint_pair_accuracy',
    'exact_two_star_reconstruction_rate',
    'syntax_validity_rate',
    'canonical_reconstruction_exact_match_rate',
    'roundtrip_success_rate',
]]


## 5) Qualitative Samples & Log Tail


In [ ]:
qual = pd.read_csv(EVAL_CONSTRAINED_DIR / 'qualitative_samples.csv')
qual.head(20)


In [ ]:
for p in [pre_log, train_log, constrained_log, unconstrained_log]:
    print(f"\n===== {p.name} (tail) =====")
    txt = p.read_text(encoding='utf-8')
    print(txt[-2000:])
